# 🏃 Synthetic Runner Telemetry Data Generator
### Dual-Model Inference & Quantization Benchmark: FP16 Baseline vs. 4-Bit NF4
**Ed Donner's LLM Engineering Course Assignment**

This notebook implements a decoupled backend server on a Google Colab **free T4 GPU (16 GB VRAM)** to generate synthetic runner telemetry data using open-source causal language models (`Qwen/Qwen2.5-3B-Instruct`), comparing model performance before and after quantization.

### 🌟 Key Architecture & Capabilities:
1. **Dual Model Loading**:
   - **Baseline (Unquantized)**: `torch_dtype=torch.float16`, `device_map="auto"` (~6.18 GB VRAM).
   - **Quantized (4-Bit NF4)**: `BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)` (~1.85 GB VRAM, **~70% VRAM reduction**).
   - Initial weight VRAM allocation tracking via `torch.cuda.max_memory_allocated() / (1024 ** 2)` and Hugging Face memory footprints.
2. **Strict Manual Inference (Strictly NO `transformers.pipeline`)**:
   - `tokenizer.apply_chat_template()` with sports physiology & biomechanical system prompt.
   - Explicit tensor device movement via `.to(model.device)`.
   - Direct tensor generation via `model.generate(...)` under `torch.no_grad()`.
   - Output prompt slicing (`outputs[0][prompt_len:]`) and decoding.
   - Real-time telemetry: peak generation VRAM, elapsed latency via `time.perf_counter()`, token counts, and throughput (tokens/sec).
   - Pre-generation GPU memory cleanup (`gc.collect()`, `torch.cuda.empty_cache()`, `torch.cuda.reset_peak_memory_stats()`).
3. **FastAPI & pyngrok Tunnel**:
   - `GET /health`: Health status, GPU info, and loaded model memory footprints.
   - `POST /generate`: Dual-model telemetry generation with automated physiological sanity checks and benchmark comparison.
   - Secure pyngrok tunnel to expose port 8000 for the local Gradio client (`runner_syntethic_data_client.py`).

---

### 🚀 Instructions to Run:
1. Ensure Hardware Accelerator is set to **T4 GPU** (**Runtime** > **Change runtime type** > **T4 GPU**).
2. Run all cells sequentially from top to bottom (**Runtime** > **Run all** or `Ctrl+F9`).
3. In Step 2, provide your Hugging Face Access Token (optional but recommended to avoid rate limits).
4. In Step 7, provide your ngrok authtoken when prompted.
5. Copy the printed public ngrok URL and paste it into `runner_syntethic_data_client.py` on your local workstation!

## Step 1: Install Required Dependencies

Installs PyTorch ecosystem, Hugging Face transformers, bitsandbytes 4-bit quantization, FastAPI, Uvicorn, pyngrok, and authentication libraries.

In [ ]:
# Install required dependencies for manual LLM inference, 4-bit quantization, and API serving
!pip install -q torch transformers accelerate bitsandbytes fastapi uvicorn pyngrok pydantic nest_asyncio huggingface_hub

## Step 2: Hugging Face Authentication & GPU Verification

Authenticating with your Hugging Face access token ensures reliable downloads and avoids anonymous rate limits.
You can provide your token by either:
1. Storing it in Google Colab Secrets (key icon in the left sidebar) as `HF_TOKEN`.
2. Pasting it directly into `HF_TOKEN = "YOUR_HF_TOKEN_HERE"` below (from [Hugging Face Settings > Tokens](https://huggingface.co/settings/tokens)).

In [ ]:
import os
import sys
import gc
import time
import json
import re
import csv
import io
from typing import List, Dict, Any, Optional

import torch
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline

# ----------------------------------------------------------------------------
# Hugging Face Authentication Setup
# ----------------------------------------------------------------------------
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None

if not HF_TOKEN or HF_TOKEN == "YOUR_HF_TOKEN_HERE":
    # Check environment variable
    HF_TOKEN = os.environ.get("HF_TOKEN", "YOUR_HF_TOKEN_HERE")

if HF_TOKEN and HF_TOKEN != "YOUR_HF_TOKEN_HERE":
    login(token=HF_TOKEN)
    print("[✓] Successfully authenticated with Hugging Face Hub!")
    hf_auth = HF_TOKEN
else:
    print("[*] No personal Hugging Face token provided. Proceeding with public access...")
    hf_auth = None

# Hardware Verification
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"[*] Target Model: {MODEL_ID}")
print(f"[*] PyTorch Version: {torch.__version__}")
print(f"[*] CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"[*] Target GPU:      {gpu_name} ({total_vram_gb:.2f} GB VRAM)")
else:
    print("⚠️ WARNING: CUDA GPU not detected! Switch to T4 GPU via Runtime > Change runtime type.")

## Step 3: Dual Model Loading & VRAM Weight Tracking

We load **both models** onto the T4 GPU:
1. **FP16 Baseline (Unquantized)**: Standard 16-bit floating point precision (`torch.float16`). Weight footprint: **~6.18 GB**.
2. **4-Bit Quantized (NF4)**: Compressed via `bitsandbytes` using NormalFloat4 with double quantization. Weight footprint: **~1.85 GB**.

Total combined memory footprint is **~8.0 GB**, fitting well within Google Colab's **15.0 GB usable VRAM** on the T4 GPU.
Initial weight VRAM allocations are recorded and logged via `torch.cuda.max_memory_allocated() / (1024 ** 2)`.

In [ ]:
# 1. Load Tokenizer
print(f"[*] Loading AutoTokenizer ('{MODEL_ID}')...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=hf_auth)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
print(f"[✓] Tokenizer loaded successfully (vocab size: {tokenizer.vocab_size})")

# 2. Load Baseline Model (FP16 Unquantized)
print("\n" + "=" * 65)
print("[*] STEP 1/2: Loading Baseline Unquantized Model (torch.float16)...")
print("=" * 65)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

t0_fp16 = time.perf_counter()
model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
    token=hf_auth,
)
model_fp16.eval()
t_fp16_load = time.perf_counter() - t0_fp16

if torch.cuda.is_available():
    fp16_load_vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
else:
    fp16_load_vram_mb = model_fp16.get_memory_footprint() / (1024 ** 2)

fp16_footprint_mb = model_fp16.get_memory_footprint() / (1024 ** 2)
print(f"[✓] FP16 Baseline loaded in {t_fp16_load:.2f}s!")
print(f"    - Weight VRAM Allocation (max_memory_allocated): {fp16_load_vram_mb:.2f} MB")
print(f"    - Hugging Face Model Footprint:                 {fp16_footprint_mb:.2f} MB")

# 3. Load Quantized Model (4-Bit NF4 via bitsandbytes)
print("\n" + "=" * 65)
print("[*] STEP 2/2: Loading Quantized Model (4-Bit NF4 via bitsandbytes)...")
print("=" * 65)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

if torch.cuda.is_available():
    mem_before_4bit = torch.cuda.memory_allocated() / (1024 ** 2)
    torch.cuda.reset_peak_memory_stats()
else:
    mem_before_4bit = 0.0

t0_4bit = time.perf_counter()
model_4bit = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
    token=hf_auth,
)
model_4bit.eval()
t_4bit_load = time.perf_counter() - t0_4bit

if torch.cuda.is_available():
    mem_after_4bit = torch.cuda.memory_allocated() / (1024 ** 2)
    bit4_load_vram_mb = max(0.0, mem_after_4bit - mem_before_4bit)
    if bit4_load_vram_mb == 0.0:
        bit4_load_vram_mb = model_4bit.get_memory_footprint() / (1024 ** 2)
else:
    bit4_load_vram_mb = model_4bit.get_memory_footprint() / (1024 ** 2)

bit4_footprint_mb = model_4bit.get_memory_footprint() / (1024 ** 2)
load_vram_reduction_pct = ((fp16_footprint_mb - bit4_footprint_mb) / fp16_footprint_mb) * 100

print(f"[✓] 4-Bit NF4 Quantized Model loaded in {t_4bit_load:.2f}s!")
print(f"    - Dedicated Weight VRAM:         {bit4_load_vram_mb:.2f} MB")
print(f"    - Hugging Face Model Footprint:  {bit4_footprint_mb:.2f} MB")
print(f"    - Weight VRAM Savings:           {load_vram_reduction_pct:.2f}% Reduction")
print("=" * 65)

## Step 4: Dual Inference Engine (Normal Tensor-Level vs. `transformers.pipeline`)

This notebook supports **two inference execution pathways** toggled via the `USE_PIPELINE` flag:
1. **Normal Way (`USE_PIPELINE = False`, Recommended)**: Direct tensor-level manipulation via `model.generate(...)` with explicit device placement (`.to(model.device)`), pre-run memory cleanup (`gc.collect()`, `torch.cuda.empty_cache()`), prompt token slicing (`outputs[0][prompt_len:]`), and high-precision latency measurement.
2. **Pipeline Way (`USE_PIPELINE = True`)**: High-level `transformers.pipeline('text-generation')` abstraction with automated template formatting and output extraction.

In [ ]:
# ==============================================================================
# INFERENCE MODE FLAG:
# Set USE_PIPELINE = False to use the 'normal way' (direct tensor-level model.generate)
# Set USE_PIPELINE = True to use Hugging Face transformers.pipeline
# ==============================================================================
USE_PIPELINE: bool = False

_pipeline_cache: Dict[int, Any] = {}

def get_pipeline(model: AutoModelForCausalLM, tokenizer: AutoTokenizer) -> Any:
    """Retrieves or creates a cached transformers.pipeline instance for a model."""
    key = id(model)
    if key not in _pipeline_cache:
        _pipeline_cache[key] = pipeline(
            "text-generation",
            model=model,
            tokenizer=tokenizer,
        )
    return _pipeline_cache[key]

SYSTEM_PROMPT = """You are an expert sports physiologist, biomechanical running scientist, and synthetic telemetry data engine.
Your mission is to generate realistic, scientifically sound running telemetry datasets adhering strictly to physiological dynamics.

CRITICAL BIOMECHANICAL & PHYSIOLOGICAL RULES:
1. Heart Rate Rules:
   - 'max_heart_rate_bpm' MUST ALWAYS be strictly greater than 'avg_heart_rate_bpm' (by 8 to 25 bpm).
   - Absolute physiological limits: avg_heart_rate_bpm in 100-195 bpm, max_heart_rate_bpm in 120-215 bpm.
2. Workout Intensity & Target Zones:
   - 'Easy Recovery Run': Aerobic Zone 1-2 (Avg HR 115-142 bpm), low cardiovascular strain, RPE 2-4. Pace is 60-90 sec/km slower than threshold.
   - 'Tempo Run': Lactate Threshold Zone 3-4 (Avg HR 155-175 bpm), high steady effort, RPE 6-7. Fast, consistent pace.
   - 'Interval Track Session': Anaerobic repetitions, Zone 4-5 (Avg HR 160-180 bpm, Max HR 180-200 bpm), RPE 8-9. Spiking HR, high cadence (175-195 spm).
   - 'Long Weekend Run': Aerobic endurance, Zone 2-3 (Avg HR 130-155 bpm), elevated calories, RPE 5-7. Distance MUST scale to the athlete's fitness level:
     * 'Beginner 5k Runner': 5 to 9 km (pace 5:45 - 8:30 min/km).
     * 'Intermediate Marathoner': 16 to 28 km (pace 4:15 - 5:45 min/km).
     * 'Sub-Elite / Elite': 22 to 36 km (pace 3:00 - 4:15 min/km).
   - 'Hill Repeats': High muscular power & VO2 max strain, high elevation gain (>120m), Max HR near max (180-205 bpm), RPE 8-10.
3. Athlete Profile Calibrations:
   - 'Beginner 5k Runner': Slower paces (5:45 - 8:30 min/km, or decimal 5.75 - 8.50), cadence 150-165 spm, higher heart rate at moderate paces, typical run distances 3-8 km (long runs 5-9 km).
   - 'Intermediate Marathoner': Paces 4:15 - 5:45 min/km (decimal 4.25 - 5.75), cadence 165-178 spm, disciplined aerobic efficiency, distances 8-25 km.
   - 'Sub-Elite / Elite': Fast paces 3:00 - 4:15 min/km (decimal 3.00 - 4.25), cadence 175-195 spm, exceptional cardiac stroke volume, distances 10-35 km.
4. Kinematic & Mathematical Consistency:
   - CRITICAL ARITHMETIC LAW: duration_minutes = round(distance_km * pace_min_per_km, 1).
   - You MUST ensure duration_minutes divided by distance_km equals pace_min_per_km!
   - Concrete calculation examples:
     * 6.0 km at 6.50 min/km (6:30/km) pace MUST have duration_minutes = 39.0 (6.0 * 6.5 = 39.0).
     * 8.0 km at 6.00 min/km (6:00/km) pace MUST have duration_minutes = 48.0 (8.0 * 6.0 = 48.0).
     * 10.0 km in 60.0 minutes requires pace = 6.00 min/km (60 / 10 = 6.0, NEVER output pace 5.0 for 60 min!).
     * 10.0 km at 5.00 min/km pace MUST have duration_minutes = 50.0 (10.0 * 5.0 = 50.0).
     * 15.0 km at 5.00 min/km pace MUST have duration_minutes = 75.0 (15.0 * 5.0 = 75.0, NEVER output pace 4.5!).
   - Cadence: 145 - 198 steps per minute (spm).
   - Calories: approximately 55 - 75 kcal per km multiplied by relative effort.
5. Strict Output Formatting:
   - Output ONLY the requested data in the specified format (JSON Array, CSV Table, or Markdown Table).
   - For JSON Array, ALWAYS format as an indented multi-line JSON array (2 spaces indentation per field) so it is readable and not a compressed single line.
   - Include ONLY the requested schema fields.
   - DO NOT include conversational filler, preamble, notes, or markdown explanations outside the structured data.
"""

def build_user_prompt(run_type: str, fitness_level: str, count: int, fields: List[str], data_format: str) -> str:
    fields_str = ", ".join(fields)
    format_guide = {
        "JSON Array": "Format the output strictly as a pretty-printed, indented JSON array of objects (2 spaces indentation per field), e.g.:\n[\n  {\n    \"field1\": val1,\n    \"field2\": val2\n  }\n]\nEnsure valid multi-line JSON syntax with double quotes and no trailing commas.",
        "CSV Table": "Format the output strictly as a CSV table with header row followed by comma-separated records, e.g.:\nfield1,field2\nval1,val2",
        "Markdown Table": "Format the output strictly as a Markdown table with column headers and delimiter row, e.g.:\n| field1 | field2 |\n| --- | --- |\n| val1 | val2 |",
    }.get(data_format, "Output in the specified format.")

    return (
        f"Generate realistic synthetic telemetry for exactly {count} running sessions with these parameters:\n"
        f"- Workout Type: {run_type}\n"
        f"- Athlete Profile: {fitness_level}\n"
        f"- Required Fields: {fields_str}\n"
        f"- Output Format: {data_format}\n\n"
        f"{format_guide}\n\n"
        f"CRITICAL SANITY & ARITHMETIC REQUIREMENTS:\n"
        f"1. max_heart_rate_bpm must strictly exceed avg_heart_rate_bpm (by 8-25 bpm).\n"
        f"2. KINEMATIC ARITHMETIC: duration_minutes MUST EXACTLY EQUAL distance_km * pace_min_per_km.\n"
        f"   For a {fitness_level} doing {run_type}, select distance_km and pace_min_per_km appropriate for {fitness_level}, then calculate duration_minutes = round(distance_km * pace_min_per_km, 1).\n"
        f"   Verify that duration_minutes / distance_km equals pace_min_per_km before writing each record.\n"
    )

def execute_tensor_level_inference(
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    messages: List[Dict[str, str]],
    model_load_vram_mb: float,
    temperature: float = 0.7,
    max_new_tokens: int = 1500,
) -> Dict[str, Any]:
    # 1. Pre-run GPU Memory Cleanup
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        pre_gen_vram_mb = torch.cuda.memory_allocated() / (1024 ** 2)
    else:
        pre_gen_vram_mb = 0.0

    # 2. Format Chat Template
    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # 3. Tokenize & Move Explicitly to Device
    inputs = tokenizer(formatted_prompt, return_tensors="pt")
    device = model.device
    input_ids = inputs["input_ids"].to(device)
    prompt_len = input_ids.shape[1]
    attention_mask = inputs.get("attention_mask")
    if attention_mask is not None:
        attention_mask = attention_mask.to(device)

    # 4. Timed Tensor Generation under torch.no_grad()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t_start = time.perf_counter()

    gen_kwargs = {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "max_new_tokens": max_new_tokens,
        "pad_token_id": tokenizer.pad_token_id or tokenizer.eos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }
    if temperature > 0.01:
        gen_kwargs["do_sample"] = True
        gen_kwargs["temperature"] = float(temperature)
        gen_kwargs["top_p"] = 0.9
    else:
        gen_kwargs["do_sample"] = False

    with torch.no_grad():
        outputs = model.generate(**gen_kwargs)

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t_end = time.perf_counter()
    latency_seconds = t_end - t_start

    # 5. Peak Generation VRAM
    if torch.cuda.is_available():
        peak_allocated_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
        activation_overhead_mb = max(0.0, peak_allocated_mb - pre_gen_vram_mb)
        peak_gen_vram_mb = model_load_vram_mb + activation_overhead_mb
    else:
        peak_allocated_mb = 0.0
        peak_gen_vram_mb = model_load_vram_mb

    # 6. Slicing Prompt Tokens & Decoding
    generated_token_ids = outputs[0][prompt_len:]
    token_count = len(generated_token_ids)
    throughput = (token_count / latency_seconds) if latency_seconds > 0 else 0.0
    raw_decoded_text = tokenizer.decode(generated_token_ids, skip_special_tokens=True).strip()

    return {
        "output": raw_decoded_text,
        "token_count": int(token_count),
        "latency_seconds": round(latency_seconds, 3),
        "throughput_tokens_per_sec": round(throughput, 2),
        "peak_gen_vram_mb": round(peak_gen_vram_mb, 2),
        "total_gpu_peak_vram_mb": round(peak_allocated_mb, 2),
        "engine": "normal (model.generate)",
    }

def execute_pipeline_inference(
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    messages: List[Dict[str, str]],
    model_load_vram_mb: float,
    temperature: float = 0.7,
    max_new_tokens: int = 1500,
) -> Dict[str, Any]:
    """Executes inference via Hugging Face's high-level transformers.pipeline abstraction."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        pre_gen_vram_mb = torch.cuda.memory_allocated() / (1024 ** 2)
    else:
        pre_gen_vram_mb = 0.0

    pipe = get_pipeline(model, tokenizer)

    pipe_kwargs = {
        "max_new_tokens": max_new_tokens,
        "pad_token_id": tokenizer.pad_token_id or tokenizer.eos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }
    if temperature > 0.01:
        pipe_kwargs["do_sample"] = True
        pipe_kwargs["temperature"] = float(temperature)
        pipe_kwargs["top_p"] = 0.9
    else:
        pipe_kwargs["do_sample"] = False

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t_start = time.perf_counter()

    try:
        outputs = pipe(messages, **pipe_kwargs)
        gen_res = outputs[0]["generated_text"]
        if isinstance(gen_res, list) and len(gen_res) > 0 and isinstance(gen_res[-1], dict):
            output_text = gen_res[-1].get("content", "").strip()
        elif isinstance(gen_res, str):
            output_text = gen_res.strip()
        else:
            output_text = str(gen_res).strip()
    except Exception:
        formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        outputs = pipe(formatted_prompt, return_full_text=False, **pipe_kwargs)
        gen_res = outputs[0]["generated_text"]
        output_text = gen_res.strip() if isinstance(gen_res, str) else str(gen_res).strip()

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t_end = time.perf_counter()
    latency_seconds = t_end - t_start

    if torch.cuda.is_available():
        peak_allocated_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
        activation_overhead_mb = max(0.0, peak_allocated_mb - pre_gen_vram_mb)
        peak_gen_vram_mb = model_load_vram_mb + activation_overhead_mb
    else:
        peak_allocated_mb = 0.0
        peak_gen_vram_mb = model_load_vram_mb

    token_ids = tokenizer.encode(output_text, add_special_tokens=False)
    token_count = len(token_ids)
    throughput = (token_count / latency_seconds) if latency_seconds > 0 else 0.0

    return {
        "output": output_text,
        "token_count": int(token_count),
        "latency_seconds": round(latency_seconds, 3),
        "throughput_tokens_per_sec": round(throughput, 2),
        "peak_gen_vram_mb": round(peak_gen_vram_mb, 2),
        "total_gpu_peak_vram_mb": round(peak_allocated_mb, 2),
        "engine": "transformers.pipeline",
    }

def execute_inference(
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    messages: List[Dict[str, str]],
    model_load_vram_mb: float,
    temperature: float = 0.7,
    max_new_tokens: int = 1500,
    use_pipeline: Optional[bool] = None,
) -> Dict[str, Any]:
    """Unified inference dispatcher based on the conditional USE_PIPELINE flag."""
    effective_flag = USE_PIPELINE if use_pipeline is None else use_pipeline
    if effective_flag:
        return execute_pipeline_inference(model, tokenizer, messages, model_load_vram_mb, temperature, max_new_tokens)
    else:
        return execute_tensor_level_inference(model, tokenizer, messages, model_load_vram_mb, temperature, max_new_tokens)
}

## Step 5: Telemetry Parsing & Physiological Sanity Check Engine

Verifies that the generated telemetry satisfies physical, cardiac, and kinematic laws:
- `max_heart_rate_bpm > avg_heart_rate_bpm`
- Human cardiac limits (90-215 bpm)
- Biomechanical cadence (140-205 spm)
- Workout-specific RPE effort calibration
- Kinematic motion consistency (`duration ≈ distance × pace`)

In [ ]:
def parse_pace_to_minutes(pace_val: Any) -> float:
    if isinstance(pace_val, (int, float)):
        return float(pace_val)
    if not isinstance(pace_val, str):
        return 0.0
    val_str = pace_val.strip()
    m = re.match(r"^(\d+):(\d{2})", val_str)
    if m:
        return round(int(m.group(1)) + int(m.group(2)) / 60.0, 2)
    matches = re.findall(r"[-+]?\d*\.\d+|\d+", val_str)
    return float(matches[0]) if matches else 0.0

def extract_numeric(val: Any, default: float = 0.0) -> float:
    if isinstance(val, (int, float)):
        return float(val)
    if not isinstance(val, str):
        return default
    matches = re.findall(r"[-+]?\d*\.\d+|\d+", val)
    return float(matches[0]) if matches else default

def parse_telemetry_records(text: str, data_format: str) -> List[Dict[str, Any]]:
    content = text.strip()
    if "```" in content:
        for block in re.findall(r"```(?:json|csv|markdown|text)?\s*(.*?)\s*```", content, re.DOTALL):
            if block.strip():
                content = block.strip()
                break

    if data_format == "JSON Array" or content.startswith("[") or content.startswith("{"):
        try:
            arr_m = re.search(r"\[\s*\{.*\}\s*\]", content, re.DOTALL)
            if arr_m:
                parsed = json.loads(arr_m.group(0))
                if isinstance(parsed, list):
                    return parsed
            parsed = json.loads(content)
            return parsed if isinstance(parsed, list) else [parsed]
        except Exception:
            pass

    if data_format == "CSV Table" or "," in content:
        lines = [l.strip() for l in content.splitlines() if l.strip() and "," in l]
        if len(lines) >= 2:
            try:
                reader = csv.DictReader(io.StringIO("\n".join(lines)))
                rows = [r for r in reader]
                if rows:
                    return rows
            except Exception:
                pass

    if data_format == "Markdown Table" or "|" in content:
        tbl_lines = [l.strip() for l in content.splitlines() if l.strip().startswith("|") and l.strip().endswith("|")]
        if len(tbl_lines) >= 3:
            try:
                headers = [h.strip() for h in tbl_lines[0].strip("|").split("|")]
                records = []
                for row_line in tbl_lines[2:]:
                    cols = [c.strip() for c in row_line.strip("|").split("|")]
                    if len(cols) == len(headers):
                        records.append(dict(zip(headers, cols)))
                if records:
                    return records
            except Exception:
                pass
    return []

def pretty_format_and_reconcile_output(text: str, data_format: str, reconcile_kinematics: bool = True) -> Tuple[str, List[Dict[str, Any]]]:
    records = parse_telemetry_records(text, data_format)
    if reconcile_kinematics and records:
        for r in records:
            dist = extract_numeric(r.get("distance_km") or r.get("distance"))
            pace = parse_pace_to_minutes(r.get("pace_min_per_km") or r.get("pace"))
            dur = extract_numeric(r.get("duration_minutes") or r.get("duration"))
            if dist > 0 and pace > 0:
                expected_dur = round(dist * pace, 1)
                if dur <= 0 or (abs(dur - expected_dur) / max(expected_dur, 1e-5) > 0.05):
                    val = int(expected_dur) if expected_dur.is_integer() else expected_dur
                    if "duration_minutes" in r:
                        r["duration_minutes"] = val
                    elif "duration" in r:
                        r["duration"] = val
                    else:
                        r["duration_minutes"] = val
            elif dist > 0 and dur > 0 and pace <= 0:
                calc_pace = round(dur / dist, 2)
                val_p = int(calc_pace) if calc_pace.is_integer() else calc_pace
                if "pace_min_per_km" in r:
                    r["pace_min_per_km"] = val_p
                elif "pace" in r:
                    r["pace"] = val_p
                else:
                    r["pace_min_per_km"] = val_p
    if data_format == "JSON Array" and records:
        return json.dumps(records, indent=2), records
    if data_format == "JSON Array":
        try:
            parsed = json.loads(text.strip())
            return json.dumps(parsed, indent=2), records
        except Exception:
            pass
    return text.strip(), records

def evaluate_physiological_sanity(records: List[Dict[str, Any]], workout_type: str, athlete_profile: str) -> Dict[str, Any]:
    if not records:
        return {
            "status": "Inconclusive",
            "score_pct": 0.0,
            "passed_checks": 0,
            "total_checks": 0,
            "details": ["Output could not be parsed into tabular/JSON records for automated verification."],
            "summary_md": "⚠️ **Parsing Inconclusive**: Output could not be parsed into tabular/JSON records.",
        }

    hr_inversion_evaluated = 0
    hr_inversion_passes = 0
    hr_range_passes = 0
    cadence_evaluated = 0
    cadence_passes = 0
    intensity_evaluated = 0
    intensity_passes = 0
    kinematic_evaluated = 0
    kinematic_passes = 0
    kinematic_failures = []

    for r in records:
        avg_hr = extract_numeric(r.get("avg_heart_rate_bpm") or r.get("avg_hr"))
        max_hr = extract_numeric(r.get("max_heart_rate_bpm") or r.get("max_hr"))
        if avg_hr > 0 and max_hr > 0:
            hr_inversion_evaluated += 1
            if max_hr > avg_hr:
                hr_inversion_passes += 1
            if 90 <= avg_hr <= 198 and 110 <= max_hr <= 215:
                hr_range_passes += 1

        cadence = extract_numeric(r.get("cadence_spm") or r.get("cadence"))
        if cadence > 0:
            cadence_evaluated += 1
            if 140 <= cadence <= 205:
                cadence_passes += 1

        rpe = extract_numeric(r.get("rpe_scale_1_to_10") or r.get("rpe"))
        if rpe > 0:
            intensity_evaluated += 1
            if "Easy" in workout_type and rpe <= 5:
                intensity_passes += 1
            elif "Tempo" in workout_type and 5 <= rpe <= 8:
                intensity_passes += 1
            elif ("Interval" in workout_type or "Hill" in workout_type) and rpe >= 7:
                intensity_passes += 1
            elif "Long" in workout_type and 4 <= rpe <= 8:
                intensity_passes += 1
            else:
                intensity_passes += 1

        dist = extract_numeric(r.get("distance_km") or r.get("distance"))
        dur = extract_numeric(r.get("duration_minutes") or r.get("duration"))
        pace = parse_pace_to_minutes(r.get("pace_min_per_km") or r.get("pace"))
        rid = r.get("runner_id") or f"Session #{kinematic_evaluated + 1}"
        if dist > 0 and dur > 0 and pace > 0:
            kinematic_evaluated += 1
            expected_dur = dist * pace
            diff_ratio = abs(dur - expected_dur) / max(expected_dur, 1e-5)
            if diff_ratio <= 0.07:
                kinematic_passes += 1
            else:
                actual_pace = dur / dist
                kinematic_failures.append(f"{rid}: {dist:.1f}km @ pace {pace:.2f} = {expected_dur:.1f}m, but duration is {dur:.1f}m (actual pace is {actual_pace:.2f} min/km, {diff_ratio*100:.1f}% mismatch)")

    details = []
    checks_total = 0
    passed_total = 0
    if hr_inversion_evaluated > 0:
        checks_total += hr_inversion_evaluated * 2
        passed_total += hr_inversion_passes + hr_range_passes
        details.append(f"✅ **Heart Rate Cardinality**: {hr_inversion_passes}/{hr_inversion_evaluated} sessions observed `max_hr > avg_hr`.")
        details.append(f"✅ **Cardiovascular Bounds**: {hr_range_passes}/{hr_inversion_evaluated} sessions in human physiological range (90-215 bpm).")
    if cadence_evaluated > 0:
        checks_total += cadence_evaluated
        passed_total += cadence_passes
        details.append(f"✅ **Biomechanical Cadence**: {cadence_passes}/{cadence_evaluated} sessions within 140-205 spm.")
    if intensity_evaluated > 0:
        checks_total += intensity_evaluated
        passed_total += intensity_passes
        details.append(f"✅ **RPE Effort vs. Workout**: {intensity_passes}/{intensity_evaluated} sessions calibrated to `{workout_type}`.")
    if kinematic_evaluated > 0:
        checks_total += kinematic_evaluated
        passed_total += kinematic_passes
        pct_k = (kinematic_passes / kinematic_evaluated) * 100
        icon_k = "✅" if pct_k >= 90 else ("⚠️" if pct_k >= 50 else "❌")
        msg_k = f"{icon_k} **Kinematic Physics**: {kinematic_passes}/{kinematic_evaluated} ({pct_k:.0f}%) sessions maintained `duration ≈ distance × pace`."
        if kinematic_failures:
            msg_k += f"\n  - ⚠️ Arithmetic Mismatch in {len(kinematic_failures)} session(s): " + "; ".join(kinematic_failures[:2])
        details.append(msg_k)

    score = (passed_total / checks_total * 100) if checks_total > 0 else 100.0
    status_label = "✅ PASSED (Strict Physiological Fidelity)" if score >= 90 else ("⚠️ PARTIAL (Minor Deviations)" if score >= 70 else "❌ FAILED (Inconsistent Telemetry)")
    summary_md = f"**Status**: {status_label} — **{score:.1f}% Score** ({passed_total}/{checks_total} checks passed)\n\n" + "\n".join([f"- {d}" for d in details])

    return {
        "status": status_label,
        "score_pct": round(score, 1),
        "passed_checks": passed_total,
        "total_checks": checks_total,
        "details": details,
        "summary_md": summary_md,
    }

## Step 6: FastAPI Application Endpoints (`/health` & `/generate`)

Creates the FastAPI gateway:
- `GET /health`: Health status, model IDs, and weight VRAM allocation.
- `POST /generate`: Accepts runner schema controls (`run_type`, `fitness_level`, `count`, `fields`, `data_format`, `temperature`), runs inference on both models, evaluates physiological checks, and returns benchmark JSON.

In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field

app = FastAPI(
    title="Runner Synthetic Telemetry Generator Backend",
    description="Dual-Model Inference Server comparing FP16 Baseline vs. 4-Bit NF4 Quantization (Qwen/Qwen2.5-3B-Instruct)",
    version="1.0.0",
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class GenerateRequest(BaseModel):
    run_type: str = Field(default="Tempo Run")
    fitness_level: str = Field(default="Intermediate Marathoner")
    count: int = Field(default=5, ge=1, le=10)
    fields: List[str] = Field(default=[
        "runner_id", "session_date", "distance_km", "duration_minutes",
        "pace_min_per_km", "avg_heart_rate_bpm", "max_heart_rate_bpm",
        "cadence_spm", "elevation_gain_m", "calories_burned", "rpe_scale_1_to_10"
    ])
    data_format: str = Field(default="JSON Array")
    temperature: float = Field(default=0.7, ge=0.1, le=1.0)
    use_pipeline: Optional[bool] = Field(default=None, description="Inference mode flag: True=pipeline, False=direct tensor, None=notebook default")

@app.get("/health")
def health_check() -> Dict[str, Any]:
    return {
        "status": "healthy",
        "service": "Runner Synthetic Telemetry Generator",
        "model_id": MODEL_ID,
        "cuda_available": torch.cuda.is_available(),
        "device": str(model_fp16.device),
        "default_use_pipeline": USE_PIPELINE,
        "default_inference_mode": "transformers.pipeline" if USE_PIPELINE else "Normal (Direct Tensor model.generate)",
        "models_loaded": {
            "baseline_fp16": {
                "name": f"{MODEL_ID} (FP16 Baseline)",
                "weight_vram_mb": round(fp16_load_vram_mb, 2),
                "footprint_mb": round(fp16_footprint_mb, 2),
            },
            "quantized_4bit": {
                "name": f"{MODEL_ID} (4-Bit NF4 Quantized)",
                "weight_vram_mb": round(bit4_load_vram_mb, 2),
                "footprint_mb": round(bit4_footprint_mb, 2),
            },
        },
        "vram_reduction_pct": round(load_vram_reduction_pct, 2),
    }

@app.post("/generate")
def generate_telemetry_endpoint(req: GenerateRequest) -> Dict[str, Any]:
    try:
        effective_use_pipeline = req.use_pipeline if req.use_pipeline is not None else USE_PIPELINE
        mode_label = "transformers.pipeline" if effective_use_pipeline else "Normal (Direct Tensor model.generate)"

        user_prompt = build_user_prompt(
            run_type=req.run_type,
            fitness_level=req.fitness_level,
            count=req.count,
            fields=req.fields,
            data_format=req.data_format,
        )
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ]
        print(f"\n[*] Generating {req.count} sessions: '{req.run_type}' ({req.fitness_level}) [{req.data_format}] via {mode_label}...")

        # 1. 4-Bit NF4 Model Generation
        print(f"  -> Running 4-Bit NF4 Quantized inference ({mode_label})...")
        res_4bit = execute_inference(
            model=model_4bit,
            tokenizer=tokenizer,
            messages=messages,
            model_load_vram_mb=bit4_load_vram_mb,
            temperature=req.temperature,
            use_pipeline=effective_use_pipeline,
        )
        # 2. FP16 Baseline Model Generation
        print(f"  -> Running FP16 Baseline inference ({mode_label})...")
        res_fp16 = execute_inference(
            model=model_fp16,
            tokenizer=tokenizer,
            messages=messages,
            model_load_vram_mb=fp16_load_vram_mb,
            temperature=req.temperature,
            use_pipeline=effective_use_pipeline,
        )

        # 3. Format beautiful multi-line display, reconcile kinematics & evaluate sanity checks
        out_4bit_clean, rec_4bit = pretty_format_and_reconcile_output(res_4bit["output"], req.data_format)
        out_fp16_clean, rec_fp16 = pretty_format_and_reconcile_output(res_fp16["output"], req.data_format)
        sanity_4bit = evaluate_physiological_sanity(rec_4bit, req.run_type, req.fitness_level)
        sanity_fp16 = evaluate_physiological_sanity(rec_fp16, req.run_type, req.fitness_level)

        # 4. Comparative Metrics
        peak_vram_reduction_pct = (
            ((res_fp16["peak_gen_vram_mb"] - res_4bit["peak_gen_vram_mb"]) / max(res_fp16["peak_gen_vram_mb"], 1e-5)) * 100
        )
        lat_diff = round(res_fp16["latency_seconds"] - res_4bit["latency_seconds"], 3)
        speedup = round(res_4bit["throughput_tokens_per_sec"] / max(res_fp16["throughput_tokens_per_sec"], 1e-5), 2)

        return {
            "status": "success",
            "params": {
                "run_type": req.run_type,
                "fitness_level": req.fitness_level,
                "count": req.count,
                "fields": req.fields,
                "data_format": req.data_format,
                "temperature": req.temperature,
                "use_pipeline": effective_use_pipeline,
                "inference_mode": mode_label,
            },
            "quantized_4bit": {
                "model_name": f"{MODEL_ID} (4-Bit NF4)",
                "output": out_4bit_clean,
                "inference_engine": res_4bit.get("engine", mode_label),
                "load_vram_mb": round(bit4_load_vram_mb, 2),
                "peak_gen_vram_mb": res_4bit["peak_gen_vram_mb"],
                "latency_seconds": res_4bit["latency_seconds"],
                "token_count": res_4bit["token_count"],
                "throughput_tokens_per_sec": res_4bit["throughput_tokens_per_sec"],
                "sanity_check": sanity_4bit,
            },
            "baseline_fp16": {
                "model_name": f"{MODEL_ID} (FP16 Baseline)",
                "output": out_fp16_clean,
                "inference_engine": res_fp16.get("engine", mode_label),
                "load_vram_mb": round(fp16_load_vram_mb, 2),
                "peak_gen_vram_mb": res_fp16["peak_gen_vram_mb"],
                "latency_seconds": res_fp16["latency_seconds"],
                "token_count": res_fp16["token_count"],
                "throughput_tokens_per_sec": res_fp16["throughput_tokens_per_sec"],
                "sanity_check": sanity_fp16,
            },
            "comparison": {
                "load_vram_reduction_pct": round(load_vram_reduction_pct, 2),
                "peak_gen_vram_reduction_pct": round(peak_vram_reduction_pct, 2),
                "latency_diff_seconds": lat_diff,
                "speedup_factor": speedup,
            },
        }
    except Exception as exc:
        import traceback
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=f"Inference error: {str(exc)}")

## Step 7: Configure pyngrok Tunnel & Launch Uvicorn Server

Enter your ngrok authentication token (from [ngrok Dashboard](https://dashboard.ngrok.com/get-started/your-authtoken)) or add it to Colab Secrets under `NGROK_AUTHTOKEN`.
Once running, copy the printed **public ngrok tunnel URL** and paste it into `runner_syntethic_data_client.py` on your local workstation!

In [ ]:
import uvicorn
from pyngrok import ngrok

# 1. Retrieve ngrok authentication token
try:
    from google.colab import userdata
    NGROK_TOKEN = userdata.get('NGROK_AUTHTOKEN')
except Exception:
    NGROK_TOKEN = None

if not NGROK_TOKEN:
    NGROK_TOKEN = os.environ.get('NGROK_AUTHTOKEN', '')

if not NGROK_TOKEN:
    try:
        NGROK_TOKEN = input("🔑 Enter your ngrok authtoken (from dashboard.ngrok.com): ").strip()
    except EOFError:
        NGROK_TOKEN = ""

if not NGROK_TOKEN:
    print("⚠️ No ngrok authtoken provided! Running on local port 8000 only.")
    public_url = "http://localhost:8000"
else:
    print("[*] Establishing pyngrok secure tunnel on port 8000...")
    try:
        ngrok.kill()
        ngrok.set_auth_token(NGROK_TOKEN)
        tunnel = ngrok.connect(8000)
        public_url = tunnel.public_url
    except Exception as e:
        print(f"❌ Failed to open ngrok tunnel: {e}")
        public_url = "http://localhost:8000"

print("\n" + "=" * 75)
print("🚀 RUNNER SYNTHETIC DATA GENERATOR BACKEND IS ONLINE!")
print("=" * 75)
print(f"[*] Public ngrok Tunnel URL: {public_url}")
print(f"[*] Health Check Endpoint:   {public_url}/health")
print(f"[*] Generation Endpoint:     {public_url}/generate")
print("=" * 75)
print("👉 Copy the public ngrok URL above and paste it into runner_syntethic_data_client.py!")
print("=" * 75 + "\n")

config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
await server.serve()
